# Kurumsal Karbon Stres Testi - ARIMAX + MLP Hibrit
Bu notebook 5 modüllü boru hattını Google Colab/Jupyter formatında uygular.

## Modül 1 - Veri Çekme, ffill/bfill, MinMax, ADF Testi

In [ ]:
from src.cbam_hybrid import fetch_market_data, clean_and_scale_data, enforce_stationarity

raw_df = fetch_market_data(period='5y')
scaled_df, scaler = clean_and_scale_data(raw_df)
stationary_df, stationarity = enforce_stationarity(scaled_df)
for col, is_stat in stationarity.items():
    print(f'{col} durağan mı? {"Evet" if is_stat else "Hayır (diff uygulandı)"}')

## Modül 2 - Doğrusal Aşama (ARIMAX)

In [ ]:
from src.cbam_hybrid import train_test_split_time_series, fit_arimax

train_df, test_df = train_test_split_time_series(stationary_df, train_ratio=0.8)
y_train, y_test = train_df['Y_EREGL'], test_df['Y_EREGL']
x1_train, x1_test = train_df['X1_KEA'], test_df['X1_KEA']
x2_train, x2_test = train_df['X2_TIO'], test_df['X2_TIO']

arimax_fit = fit_arimax(y_train, x2_train)
arimax_test_pred = arimax_fit.get_forecast(steps=len(y_test), exog=x2_test).predicted_mean
arimax_test_pred.head()

## Modül 3 - ARIMAX Hata (Residual) Çıkarımı

In [ ]:
fitted_train = arimax_fit.fittedvalues.reindex(y_train.index)
residuals_train = (y_train - fitted_train).dropna()
residuals_train.head()

## Modül 4 - Derin Öğrenme (TensorFlow/Keras MLP)

In [ ]:
from src.cbam_hybrid import build_residual_training_frame, train_mlp

mlp_X_train, mlp_y_train = build_residual_training_frame(x1_train.reindex(residuals_train.index), residuals_train)
mlp_model = train_mlp(mlp_X_train, mlp_y_train)

## Modül 5 - Hibrit Birleşim ve RMSE Karşılaştırması

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from src.cbam_hybrid import forecast_mlp_residuals

mlp_residual_test_pred = forecast_mlp_residuals(
    mlp_model=mlp_model,
    x1_test=x1_test,
    last_train_residual=float(residuals_train.iloc[-1]),
)

y_final_pred = arimax_test_pred.add(mlp_residual_test_pred, fill_value=0.0)
arimax_rmse = np.sqrt(mean_squared_error(y_test, arimax_test_pred))
hybrid_rmse = np.sqrt(mean_squared_error(y_test, y_final_pred.reindex(y_test.index)))

print(f'ARIMAX RMSE : {arimax_rmse:.6f}')
print(f'Hibrit RMSE : {hybrid_rmse:.6f}')